# AI Image Detection - Training on Google Colab

**Steps:**
1. Enable GPU: Runtime → Change runtime type → GPU (T4)
2. Run all cells in order
3. Training takes ~12 hours (free on Colab)
4. Download checkpoints at the end

In [ ]:
# Clone repository
!git clone https://github.com/YOUR_USERNAME/techjam_aigenimagedetector.git
%cd techjam_aigenimagedetector/.worktrees/feature/ai-image-detection

In [ ]:
# Install dependencies
!pip install -q torch torchvision timm opencv-python scikit-image scikit-learn tqdm matplotlib seaborn kaggle

In [ ]:
# Setup Kaggle credentials
from google.colab import files
import os

print("Upload your kaggle.json file:")
uploaded = files.upload()

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# Download CIFAKE dataset
!mkdir -p data/raw
%cd data/raw
!kaggle datasets download -d birdy654/cifake-real-and-ai-generated-synthetic-images
!unzip -q cifake-real-and-ai-generated-synthetic-images.zip -d CIFAKE
%cd ../..

In [ ]:
# Organize data
!mkdir -p data/processed/real data/processed/fake
!cp -r data/raw/CIFAKE/train/REAL/* data/processed/real/
!cp -r data/raw/CIFAKE/train/FAKE/* data/processed/fake/
!cp -r data/raw/CIFAKE/test/REAL/* data/processed/real/
!cp -r data/raw/CIFAKE/test/FAKE/* data/processed/fake/

# Verify
import os
real_count = len(os.listdir('data/processed/real'))
fake_count = len(os.listdir('data/processed/fake'))
print(f"✓ Real images: {real_count}")
print(f"✓ Fake images: {fake_count}")
print(f"✓ Total: {real_count + fake_count}")

In [ ]:
# Verify GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️ GPU not enabled! Go to Runtime → Change runtime type → GPU")

## Phase 1: Train Frequency Detector (~2 hours)

In [ ]:
!python training/train_frequency.py

## Phase 2: Train Spatial Detector (~8 hours)

In [ ]:
!python training/train_spatial.py

## Phase 3: Train Fusion Model (~30 min)

In [ ]:
!python training/train_fusion.py

## Evaluate Results

In [ ]:
!python evaluation/evaluate.py

## Download Trained Models

In [ ]:
# Zip all checkpoints
!zip -r trained_models.zip checkpoints/

# Download
from google.colab import files
files.download('trained_models.zip')

print("✅ Download complete! Extract locally and use inference.py")